# Lesson 27: Recommender Systems

*ML Academy — companion notebook*

Explicit and implicit feedback, bias baselines, neighbour-based collaborative filtering, matrix factorisation, ranking metrics, and cold start.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-27.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 2. Explicit and implicit feedback

In [1]:
import numpy as np
import pandas as pd

# 300 users, 60 films. Each user and film has 3 hidden "tastes"; users rate about 15% of films
rng = np.random.default_rng(0)
n_users, n_items, k = 300, 60, 3
U, V = rng.normal(size=(n_users, k)), rng.normal(size=(n_items, k))
popularity = rng.normal(0, 0.5, n_items)
true = 3.2 + U @ V.T * 0.6 + popularity + rng.normal(0, 0.4, (n_users, n_items))
R = np.clip(np.round(true), 1, 5)
mask = rng.random((n_users, n_items)) < 0.15
ratings = pd.DataFrame([(u, i, R[u, i]) for u, i in zip(*np.nonzero(mask))], columns=["user", "item", "rating"])
print(ratings.head(4).to_string(index=False))
print("ratings:", len(ratings), " share of the matrix filled:", round(mask.mean(), 3))

 user  item  rating
    0     0     3.0
    0     2     2.0
    0    10     4.0
    0    21     4.0
ratings: 2747  share of the matrix filled: 0.153


## 3. Baselines: averages and biases

In [2]:
# Hold out one rating per user (for evaluation) and train on the rest
test = ratings.groupby("user").sample(1, random_state=0)
train = ratings.drop(test.index)
mu = train.rating.mean()
item_mean = train.groupby("item").rating.mean()
user_bias = (train.rating - mu).groupby(train.user).mean()
rmse = lambda p: np.sqrt(np.mean((test.rating.to_numpy() - p) ** 2))
print("global mean      RMSE:", round(rmse(np.full(len(test), mu)), 3))
print("item average     RMSE:", round(rmse(test.item.map(item_mean).fillna(mu).to_numpy()), 3))
print("item + user bias RMSE:", round(rmse((test.item.map(item_mean).fillna(mu) + test.user.map(user_bias).fillna(0)).to_numpy()), 3))

global mean      RMSE: 1.077
item average     RMSE: 1.024
item + user bias RMSE: 1.057


## 4. Collaborative filtering: neighbours

In [3]:
# Item-item collaborative filtering: films are similar if the same users rate them alike
M = np.full((n_users, n_items), np.nan)
M[train.user, train.item] = train.rating
centred = M - np.nanmean(M, axis=1, keepdims=True)            # remove each user's personal scale
C = np.nan_to_num(centred)
norms = np.linalg.norm(C, axis=0) + 1e-9
S = (C.T @ C) / np.outer(norms, norms)                         # cosine similarity between films
np.fill_diagonal(S, 0)

def predict_item_item(u, i, k=10):
    rated = np.flatnonzero(~np.isnan(M[u]))
    top = rated[np.argsort(S[i, rated])[::-1][:k]]
    w = S[i, top]
    base = np.nanmean(M[u])
    return base + (w @ centred[u, top]) / (np.abs(w).sum() + 1e-9) if len(top) else mu

p_ii = np.array([predict_item_item(u, i) for u, i in zip(test.user, test.item)])
print("item-item CF     RMSE:", round(rmse(p_ii), 3))

item-item CF     RMSE: 1.012


## 6. Matrix factorisation

In [4]:
# Matrix factorisation with SGD: learn a vector for every user and film so that rating ≈ mu + b_u + b_i + p_u · q_i
def factorise(train, k=3, epochs=40, lr=0.02, reg=0.05, seed=0):
    g = np.random.default_rng(seed)
    P, Q = g.normal(0, 0.1, (n_users, k)), g.normal(0, 0.1, (n_items, k))
    bu, bi = np.zeros(n_users), np.zeros(n_items)
    data = train[["user", "item", "rating"]].to_numpy()
    for ep in range(epochs):
        for u, i, r in data[g.permutation(len(data))]:
            u, i = int(u), int(i)
            e = r - (mu + bu[u] + bi[i] + P[u] @ Q[i])
            bu[u] += lr * (e - reg * bu[u]); bi[i] += lr * (e - reg * bi[i])
            P[u], Q[i] = P[u] + lr * (e * Q[i] - reg * P[u]), Q[i] + lr * (e * P[u] - reg * Q[i])
    return P, Q, bu, bi

P, Q, bu, bi = factorise(train)
p_mf = mu + bu[test.user] + bi[test.item] + np.sum(P[test.user] * Q[test.item], axis=1)
print("matrix factorisation RMSE:", round(rmse(p_mf), 3))

matrix factorisation RMSE: 0.715


## 7. Evaluating recommendations: rankings, not ratings

In [5]:
# Ranking quality: of the 10 unseen films we recommend, how many would the user really like (true rating 4 or 5)?
# (We can check this only because the data is simulated and we know every true rating.)
liked = R >= 4
def precision_at_10(score_fn):
    hits = []
    for u in range(n_users):
        scores = score_fn(u).astype(float).copy()
        scores[mask[u]] = -np.inf                                   # don't recommend films they've rated
        top = np.argsort(scores)[::-1][:10]
        hits.append(liked[u, top].mean())
    return np.mean(hits)

pop = train.groupby("item").rating.mean().reindex(range(n_items), fill_value=mu).to_numpy()
rand = np.random.default_rng(1)
print("precision@10, random films:          ", round(precision_at_10(lambda u: rand.random(n_items)), 3))
print("precision@10, highest-rated films:   ", round(precision_at_10(lambda u: pop), 3))
print("precision@10, matrix factorisation:  ", round(precision_at_10(lambda u: bi + Q @ P[u]), 3))

precision@10, random films:           0.447
precision@10, highest-rated films:    0.671
precision@10, matrix factorisation:   0.912


## 9. In Python: content-based recommendations

In [6]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("../data/houses.csv")   # download it from Lesson 1
T = TfidfVectorizer().fit_transform(df["description"])
sim = cosine_similarity(T) + 0.2 * (df["area"].to_numpy()[:, None] == df["area"].to_numpy()[None, :])
np.fill_diagonal(sim, -1)                         # never recommend the house itself
viewed = 5                                         # a buyer looked at house id 5
row = df.index[df.id == viewed][0]
top = np.argsort(sim[row])[::-1][:3]
print("viewed:", df.description[row])
print(df.loc[top, ["id", "area", "description"]].to_string(index=False))

viewed: new spacious villa with garden
 id      area                    description
 12    Suburb new spacious villa near school
  8 Outskirts    new large villa with garden
  6    Suburb     old villa needs renovation


## 12. Going deeper: alternating least squares and popularity bias

In [7]:
import numpy as np
rng = np.random.default_rng(0)
n_users, n_items, k = 200, 40, 3
P_true, Q_true = rng.normal(0, 1, (n_users, k)), rng.normal(0, 1, (n_items, k))
R = P_true @ Q_true.T + rng.normal(0, 0.3, (n_users, n_items))
mask = rng.random((n_users, n_items)) < 0.2            # only 20% of ratings observed
test = (rng.random((n_users, n_items)) < 0.25) & mask
train = mask & ~test
P, Q, lam = rng.normal(0, 0.1, (n_users, k)), rng.normal(0, 0.1, (n_items, k)), 1.0
for sweep in range(1, 9):
    for u in range(n_users):                           # fix Q, solve a small ridge problem per user
        Qi = Q[train[u]]
        P[u] = np.linalg.solve(Qi.T @ Qi + lam * np.eye(k), Qi.T @ R[u, train[u]])
    for i in range(n_items):                           # fix P, solve per item
        Pu = P[train[:, i]]
        Q[i] = np.linalg.solve(Pu.T @ Pu + lam * np.eye(k), Pu.T @ R[train[:, i], i])
    if sweep in (1, 2, 4, 8):
        rmse = np.sqrt(np.mean((R - P @ Q.T)[test] ** 2))
        print("sweep", sweep, " test RMSE", round(rmse, 3))

sweep 1  test RMSE 1.635
sweep 2  test RMSE 1.456
sweep 4  test RMSE 0.964
sweep 8  test RMSE 0.762


In [8]:
popularity = train.sum(axis=0)                         # how many users interacted with each item
scores_pop = np.tile(popularity, (n_users, 1)).astype(float)
scores_mf = P @ Q.T
for name, S in [("popular", scores_pop), ("matrix factorisation", scores_mf)]:
    S = np.where(train, -np.inf, S)                    # don't recommend items already seen
    top = np.argsort(-S, axis=1)[:, :5]
    coverage = len(np.unique(top)) / n_items
    avg_pop = popularity[top].mean()
    print(f"{name:21s} catalogue coverage {coverage:.2f}   average popularity of recommended items {avg_pop:.1f}")
print("average popularity across all items:", round(popularity.mean(), 1))

popular               catalogue coverage 0.30   average popularity of recommended items 38.7
matrix factorisation  catalogue coverage 0.85   average popularity of recommended items 30.4
average popularity across all items: 29.2


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [9]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
cols = ["size_m2", "bedrooms", "dist_km", "age_yrs", "price_lakh"]
Z = StandardScaler().fit_transform(df[cols])
sim = cosine_similarity(Z)
np.fill_diagonal(sim, -1)
for viewed in [2, 11]:
    row = df.index[df.id == viewed][0]
    top = np.argsort(sim[row])[::-1][:3]
    print("viewed", viewed, "→ similar:", df.id.iloc[top].tolist(), " similarity", np.round(sim[row, top], 2).tolist())

viewed 2 → similar: [11, 1, 3]  similarity [0.69, 0.68, 0.33]
viewed 11 → similar: [1, 4, 2]  similarity [0.86, 0.74, 0.69]


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [10]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: A bias baseline

Predict a rating as the global mean plus the item's and the user's deviations.

In [11]:
mu, item_avg, user_avg = 3.5, 4.2, 3.1
pred = mu  # ← add the item and user deviations
check(pred, 3.8000000000000003, hint='Add each deviation from the global mean.')

❌ Not yet. Hint: Add each deviation from the global mean.


### Challenge 2: Precision at k

Compute precision@5 for these recommendations.

In [12]:
recommended = [4, 9, 1, 7, 3]
liked = {1, 2, 3, 8}
result = len(liked) / 5  # ← count only the hits
check(result, 0.4, hint='Count recommended items that are in liked, divided by k.')

❌ Not yet. Hint: Count recommended items that are in liked, divided by k.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. A bias baseline** — the missing line:

```python
pred = mu + (item_avg - mu) + (user_avg - mu)
```

**2. Precision at k** — the missing line:

```python
result = len(set(recommended) & liked) / 5
```